# 📊 End-to-End Benchmarking, Results Ledger & Dossier Analytics

This tutorial demonstrates how to leverage the **NeuralOps Results Ledger** and benchmarking harness to execute benchmarks, track performance telemetry across versions, query historical records via SQLite, and generate automated module dossiers.

### Core Topics Covered:
1. **Executing Empirical Benchmarks**: Run multi-mode throughput and latency benchmarks comparing linear attention vs. naive PyTorch quadratic baselines.
2. **Persistent Telemetry Logging (`ResultsLedger`)**: Record benchmarks with comprehensive hardware metadata (SM version, arithmetic intensity, memory footprint).
3. **Interactive SQLite Ledger Querying**: Run complex SQL queries to filter Pareto-optimal configurations.
4. **Automated Architectural Dossiers**: Generate Markdown dossiers and summary cards for publishing research artifacts.
5. **Multi-Version Performance Tracking**: Compare speedups across library releases (e.g. `v1.3.6` vs `v1.3.7`).


In [1]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import torch

import neuralops
from neuralops import (
    get_default_ledger,
    ATTENTION_REGISTRY,
    COMPRESSOR_REGISTRY,
    profile_roofline,
)

print(f"NeuralOps Version: {neuralops.__version__}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


/mnt/d/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-10-17 15:51:28,969 [INFO] Registered trainer: rl -> RLTrainer


2026-10-17 15:51:28,969 [INFO] Registered trainer: trl -> RLTrainer


2026-10-17 15:51:29,024 [INFO] Registered trainer: unsloth -> UnslothTrainer


NeuralOps Version: 1.3.7
Device: NVIDIA GeForce RTX 4070 Laptop GPU


## 1. Inspecting the Persistent Ledger Index

The `ResultsLedger` maintains dual-layer storage:
- **Append-only JSONL logs**: Guaranteeing non-destructive append operations.
- **SQLite Index (`index.db`)**: Providing high-speed indexing across module names, categories, git commits, and versions.


In [2]:
ledger = get_default_ledger()

print(f"Ledger Root Directory: {ledger.root_dir}")
print(f"SQLite Path:           {ledger.sqlite_path}")
print(f"JSONL Path:            {ledger.jsonl_path}")

stats = ledger.get_summary_stats()
print("\nLedger Global Telemetry Summary:")
print(f"  Total Historical Records: {stats['total_records']}")
print(f"  Active Records:           {stats['active_records']}")
print(f"  Distinct Modules:         {stats['distinct_modules']}")
print(f"  Categories Tracked:       {stats['by_category']}")
print(f"  Versions Tracked:         {stats['by_version']}")


Ledger Root Directory: /mnt/d/SVD_linear_Attention/results
SQLite Path:           /mnt/d/SVD_linear_Attention/results/registry/master_ledger.sqlite
JSONL Path:            /mnt/d/SVD_linear_Attention/results/registry/master_ledger.jsonl

Ledger Global Telemetry Summary:
  Total Historical Records: 2438
  Active Records:           2434
  Distinct Modules:         584
  Categories Tracked:       {'ablation': 28, 'attention': 2123, 'baseline': 1, 'compressor': 210, 'cuda_kernel': 36, 'triton_kernel': 40}
  Versions Tracked:         {'v1.3.6': 28, 'v1.3.7': 2410}


## 2. Querying the SQLite Index Directly

We can leverage the high-level `ledger.query()` API or raw SQL to analyze performance across canonical attention mechanisms.


In [3]:
# Query top attention mechanisms by speedup vs PyTorch
top_attns = ledger.query(category="attention", limit=10)

records_data = []
for r in top_attns:
    speedup = r.metrics.get("speedup_x") or r.metrics.get("speedup_vs_pytorch") or 1.0
    lat = r.metrics.get("latency_triton_ms") or r.metrics.get("latency_ms") or 0.0
    mem = r.metrics.get("peak_vram_mb") or r.metrics.get("peak_memory_mb") or 0.0
    records_data.append({
        "Module": r.module_name,
        "Version": r.version_tag,
        "Latency (ms)": round(lat, 3),
        "Speedup (x)": round(speedup, 2),
        "Peak Memory (MB)": round(mem, 1),
        "Commit": r.git_commit_sha[:8] if r.git_commit_sha else "N/A",
    })

df = pd.DataFrame(records_data)
print(df.to_string(index=False))

         Module Version  Latency (ms)  Speedup (x)  Peak Memory (MB)   Commit
gated_delta_net  v1.3.7         1.302        59.06              32.2 a7621194
            gla  v1.3.7         3.074         1.00             178.9 a7621194
gated_delta_net  v1.3.7         3.036        52.69              76.4 a7621194
gated_delta_net  v1.3.7         0.884        11.76               9.5 a7621194
gated_delta_net  v1.3.7        14.855        40.75             257.3 a7621194
 gated_deltanet  v1.3.7         1.850         2.50               0.0 9d144209
mamba_svd_flash  v1.3.7       541.897         1.00               0.0 c5aaefc1
  lsh_attention  v1.3.7        13.078         1.00               0.0 c5aaefc1
      svd_mamba  v1.3.7        11.874         1.00               0.0 c5aaefc1
 nasc_attention  v1.3.7        11.080         1.00               0.0 c5aaefc1


## 3. Visualizing Speedup Distributions Across Attention Families

Let us visualize the empirical speedup distribution across benchmarked linear attention modules.


In [4]:
# Extract speedups for plotting
speedup_records = ledger.query(category="attention", limit=20)
names = []
speedups = []
for r in speedup_records:
    s = r.metrics.get("speedup_x") or r.metrics.get("speedup_vs_pytorch") or 1.0
    if s >= 1.0:
        names.append(r.module_name)
        speedups.append(s)
        if len(names) >= 8:
            break

if names:
    plt.figure(figsize=(10, 4.5), dpi=100)
    bars = plt.barh(names, speedups, color='royalblue', edgecolor='navy', height=0.6)
    plt.axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='PyTorch Reference Baseline (1.0x)')
    plt.xlabel('Measured Speedup (x)', fontsize=12)
    plt.title('Empirical Speedup Across Canonical Attention Modules', fontsize=13, fontweight='bold')
    plt.grid(axis='x', linestyle=':', alpha=0.6)
    plt.legend()
    for bar in bars:
        w = bar.get_width()
        plt.text(w + 0.5, bar.get_y() + bar.get_height()/2, f'{w:.1f}x', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient records for bar plot.")

## 4. Module Dossier Retrieval & Analysis

Each module indexed in the ledger can produce a comprehensive research dossier containing run histories, parameter settings, and hardware execution metrics.


In [5]:
# Inspect dossier for a flagship module: gated_delta_net
dossier = ledger.get_module_dossier("gated_delta_net")
if dossier:
    print(f"=== Module Dossier: {dossier.get('module_name')} ===")
    print(f"Category: {dossier.get('category')}")
    print(f"Total Historical Runs: {dossier.get('total_runs')}")
    
    latest_run = dossier.get("latest_record")
    if latest_run:
        m = latest_run.get("metrics", {})
        speedup = m.get("speedup_x") or m.get("speedup_vs_pytorch") or 1.0
        lat = m.get("latency_triton_ms") or m.get("latency_ms") or 0.0
        mem = m.get("peak_vram_mb") or m.get("peak_memory_mb") or 0.0
        hw = latest_run.get("experimental_setup", {}).get("hardware", "GPU")
        print(f"\nLatest Run Details ({latest_run.get('record_id')}):")
        print(f"  Version:               {latest_run.get('version_tag')}")
        print(f"  Measured Speedup:      {speedup:.2f}x")
        print(f"  Measured Latency:      {lat:.3f} ms")
        print(f"  Peak Memory Footprint: {mem:.1f} MB")
        print(f"  Hardware Device:       {hw}")
else:
    print("Dossier for 'gated_delta_net' not found.")

=== Module Dossier: gated_delta_net ===
Category: attention
Total Historical Runs: 14

Latest Run Details (c69bccac00638cf0):
  Version:               v1.3.7
  Measured Speedup:      59.06x
  Measured Latency:      1.302 ms
  Peak Memory Footprint: 32.2 MB
  Hardware Device:       NVIDIA GeForce RTX 4070 Laptop GPU (SM 8.9)


## 5. Direct SQLite Analytical Queries

Because the index is stored in a relational SQLite table, advanced SQL aggregations and queries can be executed directly for telemetry reporting.


In [6]:
conn = sqlite3.connect(ledger.sqlite_path)
cursor = conn.cursor()

# Query average speedup and latency grouped by category from the `results` table
query = """
SELECT 
    category,
    COUNT(*) as record_count,
    ROUND(AVG(speedup_x), 2) as avg_speedup,
    ROUND(MAX(speedup_x), 2) as max_speedup,
    ROUND(AVG(latency_triton_ms), 3) as avg_triton_latency_ms
FROM results
WHERE status = 'active' AND speedup_x IS NOT NULL
GROUP BY category
ORDER BY record_count DESC;
"""

sql_df = pd.read_sql_query(query, conn)
conn.close()

print("Category Performance Aggregates from SQLite Index:")
print(sql_df.to_string(index=False))
print("\nEnd-to-End Benchmarking & Results Ledger walkthrough completed successfully [100% OK]!")

Category Performance Aggregates from SQLite Index:
     category  record_count  avg_speedup  max_speedup  avg_triton_latency_ms
triton_kernel            39        43.98       743.23                  0.720
  cuda_kernel            36      1625.34     28466.32                  0.915
    attention            31        10.40        59.06                  3.774
     ablation             1         0.68         0.68                    NaN
   compressor             1         0.01         0.01                 10.090

End-to-End Benchmarking & Results Ledger walkthrough completed successfully [100% OK]!
